# TextRank vs LexRank Evaluation using ROUGE Scores

This notebook evaluates the quality of TextRank and LexRank summaries using ROUGE (Recall-Oriented Understudy for Gisting Evaluation) metrics against ground truth summaries.

## Objective:
- Load TextRank summaries from `textrank_results/` folder
- Load LexRank summaries from `lexrank_results/` folder
- Load ground truth summaries from `ground_truth/` folder
- Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores
- Compare TextRank vs LexRank performance
- Generate evaluation report

## ROUGE Metrics:
- **ROUGE-1**: Measures overlap of unigrams (single words)
- **ROUGE-2**: Measures overlap of bigrams (word pairs)
- **ROUGE-L**: Measures longest common subsequence

## Input:
- `textrank_results/` folder containing TextRank summaries
- `lexrank_results/` folder containing LexRank summaries
- `ground_truth/` folder containing reference summaries

## Output:
- ROUGE scores comparison table
- Performance analysis and recommendations


In [1]:
pip install rouge-score

  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
Using cached absl_py-2.3.1-py3-none-any.whl (135 kB)
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25027 sha256=3616b66a23bb0185c85d3345dd1bdb777738ece8b1881b0bd0b2193fbae9fbec
  Stored in directory: c:\users\uanus\appdata\local\pip\cache\wheels\85\9d\af\01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score

   ---------------------------------------- 0/2 [absl-py]
   -------------------- ------------------- 1/2 [rouge-score]
   ---------------------------------------- 2/2 [rouge-score]

Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Building 'rouge-score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge-score'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
# Import required libraries
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ROUGE evaluation
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
    print("Using rouge_score library")
except ImportError:
    ROUGE_AVAILABLE = False
    print("rouge_score not available, will use alternative implementation")

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("Libraries imported successfully")


Using rouge_score library
Libraries imported successfully


In [8]:
# Configuration
textrank_results_path = Path("textrank_results")
lexrank_results_path = Path("lexrank_results")
ground_truth_path = Path("ground_truth")

years = ["2020", "2021", "2022", "2023", "2024"]

# Initialize ROUGE scorer
if ROUGE_AVAILABLE:
    rouge_scorer_instance = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
else:
    rouge_scorer_instance = None

print("Configuration:")
print(f"TextRank results path: {textrank_results_path}")
print(f"LexRank results path: {lexrank_results_path}")
print(f"Ground truth path: {ground_truth_path}")
print(f"Years to evaluate: {years}")
print(f"ROUGE library available: {ROUGE_AVAILABLE}")

# Check if paths exist
print(f"\nPath verification:")
print(f"TextRank results exist: {textrank_results_path.exists()}")
print(f"LexRank results exist: {lexrank_results_path.exists()}")
print(f"Ground truth exists: {ground_truth_path.exists()}")


Configuration:
TextRank results path: textrank_results
LexRank results path: lexrank_results
Ground truth path: ground_truth
Years to evaluate: ['2020', '2021', '2022', '2023', '2024']
ROUGE library available: True

Path verification:
TextRank results exist: True
LexRank results exist: True
Ground truth exists: True


In [4]:
# Load summaries and ground truth
def load_summary(file_path):
    """Load summary from file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            # Remove header lines (first 3 lines typically)
            lines = content.split('\n')
            if len(lines) > 3:
                # Skip header and get the actual summary
                summary_lines = lines[3:]
                summary = '\n'.join(summary_lines).strip()
                return summary
            return content.strip()
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return ""

def load_ground_truth():
    """Load ground truth files"""
    ground_truth_files = {}
    
    # Load GPT-4o-mini ground truth
    gpt_file = ground_truth_path / "gpt-4o-mini.txt"
    if gpt_file.exists():
        ground_truth_files['gpt-4o-mini'] = load_summary(gpt_file)
        print(f"Loaded GPT-4o-mini ground truth")
    
    # Load Groq-LLaMA ground truth
    groq_file = ground_truth_path / "groq-llama.txt"
    if groq_file.exists():
        ground_truth_files['groq-llama'] = load_summary(groq_file)
        print(f"Loaded Groq-LLaMA ground truth")
    
    return ground_truth_files

# Load all summaries
textrank_summaries = {}
lexrank_summaries = {}
ground_truth_summaries = {}

print("Loading summaries...")

# Load TextRank summaries
textrank_overall_file = textrank_results_path / "textrank_overall_summary.txt"
if textrank_overall_file.exists():
    textrank_summaries['overall'] = load_summary(textrank_overall_file)
    print(f"Loaded TextRank overall summary")

for year in years:
    textrank_file = textrank_results_path / f"textrank_summary_{year}.txt"
    if textrank_file.exists():
        textrank_summaries[year] = load_summary(textrank_file)
        print(f"Loaded TextRank summary for {year}")

# Load LexRank summaries
lexrank_overall_file = lexrank_results_path / "lexrank_overall_summary.txt"
if lexrank_overall_file.exists():
    lexrank_summaries['overall'] = load_summary(lexrank_overall_file)
    print(f"Loaded LexRank overall summary")

for year in years:
    lexrank_file = lexrank_results_path / f"lexrank_summary_{year}.txt"
    if lexrank_file.exists():
        lexrank_summaries[year] = load_summary(lexrank_file)
        print(f"Loaded LexRank summary for {year}")

# Load ground truth
ground_truth_summaries = load_ground_truth()

print(f"\nSummary loading completed:")
print(f"TextRank summaries loaded: {len(textrank_summaries)}")
print(f"LexRank summaries loaded: {len(lexrank_summaries)}")
print(f"Ground truth summaries loaded: {len(ground_truth_summaries)}")


Loading summaries...
Loaded TextRank overall summary
Loaded TextRank summary for 2020
Loaded TextRank summary for 2021
Loaded TextRank summary for 2022
Loaded TextRank summary for 2023
Loaded TextRank summary for 2024
Loaded LexRank overall summary
Loaded LexRank summary for 2020
Loaded LexRank summary for 2021
Loaded LexRank summary for 2022
Loaded LexRank summary for 2023
Loaded LexRank summary for 2024
Loaded GPT-4o-mini ground truth
Loaded Groq-LLaMA ground truth

Summary loading completed:
TextRank summaries loaded: 6
LexRank summaries loaded: 6
Ground truth summaries loaded: 2


In [5]:
# Calculate ROUGE scores
def calculate_rouge_scores_simple(predicted, reference):
    """Simple ROUGE implementation using basic text overlap"""
    if not predicted or not reference:
        return {
            'rouge1': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rouge2': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rougeL': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0}
        }
    
    # Tokenize texts
    pred_tokens = word_tokenize(predicted.lower())
    ref_tokens = word_tokenize(reference.lower())
    
    # ROUGE-1 (unigrams)
    pred_unigrams = set(pred_tokens)
    ref_unigrams = set(ref_tokens)
    
    overlap_1 = len(pred_unigrams.intersection(ref_unigrams))
    precision_1 = overlap_1 / len(pred_unigrams) if len(pred_unigrams) > 0 else 0
    recall_1 = overlap_1 / len(ref_unigrams) if len(ref_unigrams) > 0 else 0
    f1_1 = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0
    
    # ROUGE-2 (bigrams)
    pred_bigrams = set(zip(pred_tokens, pred_tokens[1:]))
    ref_bigrams = set(zip(ref_tokens, ref_tokens[1:]))
    
    overlap_2 = len(pred_bigrams.intersection(ref_bigrams))
    precision_2 = overlap_2 / len(pred_bigrams) if len(pred_bigrams) > 0 else 0
    recall_2 = overlap_2 / len(ref_bigrams) if len(ref_bigrams) > 0 else 0
    f1_2 = 2 * precision_2 * recall_2 / (precision_2 + recall_2) if (precision_2 + recall_2) > 0 else 0
    
    # ROUGE-L (simplified - using word overlap ratio)
    f1_L = f1_1  # Simplified version
    
    return {
        'rouge1': {'precision': precision_1, 'recall': recall_1, 'fmeasure': f1_1},
        'rouge2': {'precision': precision_2, 'recall': recall_2, 'fmeasure': f1_2},
        'rougeL': {'precision': precision_1, 'recall': recall_1, 'fmeasure': f1_L}
    }

def calculate_rouge_scores(predicted, reference):
    """Calculate ROUGE scores for predicted vs reference text"""
    if not predicted or not reference:
        return {
            'rouge1': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rouge2': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0},
            'rougeL': {'precision': 0.0, 'recall': 0.0, 'fmeasure': 0.0}
        }
    
    if ROUGE_AVAILABLE and rouge_scorer_instance:
        scores = rouge_scorer_instance.score(reference, predicted)
        return scores
    else:
        return calculate_rouge_scores_simple(predicted, reference)

# Calculate ROUGE scores for overall summaries against ground truth
rouge_results = []

print("Calculating ROUGE scores for overall summaries...")

# Evaluate overall summaries against both ground truth files
if 'overall' in textrank_summaries and 'overall' in lexrank_summaries:
    textrank_overall = textrank_summaries['overall']
    lexrank_overall = lexrank_summaries['overall']
    
    for gt_name, ground_truth in ground_truth_summaries.items():
        print(f"\nEvaluating against {gt_name} ground truth:")
        
        # Calculate TextRank ROUGE scores
        if textrank_overall:
            textrank_scores = calculate_rouge_scores(textrank_overall, ground_truth)
            print(f"  TextRank ROUGE-1 F1: {textrank_scores['rouge1'].fmeasure:.3f}")
            print(f"  TextRank ROUGE-2 F1: {textrank_scores['rouge2'].fmeasure:.3f}")
            print(f"  TextRank ROUGE-L F1: {textrank_scores['rougeL'].fmeasure:.3f}")
        else:
            textrank_scores = None
            print(f"  No TextRank overall summary")
        
        # Calculate LexRank ROUGE scores
        if lexrank_overall:
            lexrank_scores = calculate_rouge_scores(lexrank_overall, ground_truth)
            print(f"  LexRank ROUGE-1 F1: {lexrank_scores['rouge1'].fmeasure:.3f}")
            print(f"  LexRank ROUGE-2 F1: {lexrank_scores['rouge2'].fmeasure:.3f}")
            print(f"  LexRank ROUGE-L F1: {lexrank_scores['rougeL'].fmeasure:.3f}")
        else:
            lexrank_scores = None
            print(f"  No LexRank overall summary")
        
        # Store results
        rouge_results.append({
            'ground_truth': gt_name,
            'summary_type': 'overall',
            'textrank_rouge1': textrank_scores['rouge1'].fmeasure if textrank_scores else 0.0,
            'textrank_rouge2': textrank_scores['rouge2'].fmeasure if textrank_scores else 0.0,
            'textrank_rougeL': textrank_scores['rougeL'].fmeasure if textrank_scores else 0.0,
            'lexrank_rouge1': lexrank_scores['rouge1'].fmeasure if lexrank_scores else 0.0,
            'lexrank_rouge2': lexrank_scores['rouge2'].fmeasure if lexrank_scores else 0.0,
            'lexrank_rougeL': lexrank_scores['rougeL'].fmeasure if lexrank_scores else 0.0,
            'textrank_precision_1': textrank_scores['rouge1'].precision if textrank_scores else 0.0,
            'textrank_recall_1': textrank_scores['rouge1'].recall if textrank_scores else 0.0,
            'lexrank_precision_1': lexrank_scores['rouge1'].precision if lexrank_scores else 0.0,
            'lexrank_recall_1': lexrank_scores['rouge1'].recall if lexrank_scores else 0.0
        })

print(f"\nROUGE score calculation completed for {len(rouge_results)} evaluations")


Calculating ROUGE scores for overall summaries...

Evaluating against gpt-4o-mini ground truth:
  TextRank ROUGE-1 F1: 0.176
  TextRank ROUGE-2 F1: 0.022
  TextRank ROUGE-L F1: 0.105
  LexRank ROUGE-1 F1: 0.169
  LexRank ROUGE-2 F1: 0.000
  LexRank ROUGE-L F1: 0.073

Evaluating against groq-llama ground truth:
  TextRank ROUGE-1 F1: 0.141
  TextRank ROUGE-2 F1: 0.012
  TextRank ROUGE-L F1: 0.111
  LexRank ROUGE-1 F1: 0.157
  LexRank ROUGE-2 F1: 0.000
  LexRank ROUGE-L F1: 0.072

ROUGE score calculation completed for 2 evaluations


In [6]:
# Create comparison DataFrame and display results
if rouge_results:
    df_rouge = pd.DataFrame(rouge_results)
    
    print("\nROUGE SCORES COMPARISON TABLE")
    print("=" * 100)
    
    # Display detailed results
    print(f"\nDetailed ROUGE Scores by Ground Truth:")
    print("-" * 100)
    print(f"{'Ground Truth':<15} {'Method':<10} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10}")
    print("-" * 100)
    
    for _, row in df_rouge.iterrows():
        gt_name = row['ground_truth']
        print(f"{gt_name:<15} {'TextRank':<10} {row['textrank_rouge1']:<10.3f} {row['textrank_rouge2']:<10.3f} {row['textrank_rougeL']:<10.3f}")
        print(f"{'':<15} {'LexRank':<10} {row['lexrank_rouge1']:<10.3f} {row['lexrank_rouge2']:<10.3f} {row['lexrank_rougeL']:<10.3f}")
        print("-" * 100)
    
    # Calculate average scores across all ground truth files
    print(f"\nAVERAGE ROUGE SCORES (across all ground truth files):")
    print("-" * 60)
    print(f"{'Method':<10} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10}")
    print("-" * 60)
    
    textrank_avg_1 = df_rouge['textrank_rouge1'].mean()
    textrank_avg_2 = df_rouge['textrank_rouge2'].mean()
    textrank_avg_L = df_rouge['textrank_rougeL'].mean()
    
    lexrank_avg_1 = df_rouge['lexrank_rouge1'].mean()
    lexrank_avg_2 = df_rouge['lexrank_rouge2'].mean()
    lexrank_avg_L = df_rouge['lexrank_rougeL'].mean()
    
    print(f"{'TextRank':<10} {textrank_avg_1:<10.3f} {textrank_avg_2:<10.3f} {textrank_avg_L:<10.3f}")
    print(f"{'LexRank':<10} {lexrank_avg_1:<10.3f} {lexrank_avg_2:<10.3f} {lexrank_avg_L:<10.3f}")
    
    # Determine winner
    print(f"\nPERFORMANCE COMPARISON:")
    print("-" * 40)
    
    rouge1_winner = "TextRank" if textrank_avg_1 > lexrank_avg_1 else "LexRank"
    rouge2_winner = "TextRank" if textrank_avg_2 > lexrank_avg_2 else "LexRank"
    rougeL_winner = "TextRank" if textrank_avg_L > lexrank_avg_L else "LexRank"
    
    print(f"ROUGE-1 Winner: {rouge1_winner} ({max(textrank_avg_1, lexrank_avg_1):.3f})")
    print(f"ROUGE-2 Winner: {rouge2_winner} ({max(textrank_avg_2, lexrank_avg_2):.3f})")
    print(f"ROUGE-L Winner: {rougeL_winner} ({max(textrank_avg_L, lexrank_avg_L):.3f})")
    
    # Overall winner
    textrank_total = textrank_avg_1 + textrank_avg_2 + textrank_avg_L
    lexrank_total = lexrank_avg_1 + lexrank_avg_2 + lexrank_avg_L
    
    overall_winner = "TextRank" if textrank_total > lexrank_total else "LexRank"
    print(f"\nOverall Winner: {overall_winner}")
    print(f"TextRank Total Score: {textrank_total:.3f}")
    print(f"LexRank Total Score: {lexrank_total:.3f}")
    
else:
    print("No ROUGE scores calculated. Please check if ground truth files are available.")



ROUGE SCORES COMPARISON TABLE

Detailed ROUGE Scores by Ground Truth:
----------------------------------------------------------------------------------------------------
Ground Truth    Method     ROUGE-1    ROUGE-2    ROUGE-L   
----------------------------------------------------------------------------------------------------
gpt-4o-mini     TextRank   0.176      0.022      0.105     
                LexRank    0.169      0.000      0.073     
----------------------------------------------------------------------------------------------------
groq-llama      TextRank   0.141      0.012      0.111     
                LexRank    0.157      0.000      0.072     
----------------------------------------------------------------------------------------------------

AVERAGE ROUGE SCORES (across all ground truth files):
------------------------------------------------------------
Method     ROUGE-1    ROUGE-2    ROUGE-L   
------------------------------------------------------------
Text